### Preliminaries

In [ ]:
# Show all figures inline.  
%matplotlib inline

# Add olfaction-prediction to the Python path.  
import os
import sys
curr_path = os.getcwd()
gerkin_path = os.path.split(curr_path)[0]
olfaction_prediction_path = os.path.split(gerkin_path)[0]
sys.path.append(olfaction_prediction_path)
import opc_python

# Import numerical libraries.  
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Import generic utility modules I wrote to load the data from the tab-delimited text files and to score predictions.  
from opc_python.utils import loading, scoring

# Import the modules I wrote for actually shaping and fitting the data to the model.  
from opc_python.gerkin import dream,fit1,fit2,params

In [ ]:
other_CIDs = []
import csv
with open('/Users/rgerkin/Desktop/Odors.tsv', newline='') as f:
    reader = csv.reader(f, delimiter='\t')
    for i,row in enumerate(reader):
        if i>0:
            if row[3] != 'N/A':
                other_CIDs += [int(row[3])]
CIDs = loading.get_CIDs('training')+loading.get_CIDs('leaderboard')+loading.get_CIDs('testset')
len(CIDs),len(other_CIDs),len(set(CIDs).intersection(other_CIDs))

In [ ]:
import pandas as pd
df = pd.read_csv('../../data/3cases_Jun1.txt',delimiter='\t')
gabor_collab = df[0:21]
gabor_comp = df[22:43].set_index(pd.Index(range(21)))
rick_comp = df[44:65].set_index(pd.Index(range(21)))

In [ ]:
gabor_comp

In [ ]:
float(gabor_collab['val'][12]) + float(gabor_collab['val'][18]) + gabor_collab['val'].drop([12,18]).astype('float').mean()

In [ ]:
float(gabor_comp['val'][12]) + float(gabor_comp['val'][18]) + gabor_comp['val'].drop([12,18]).astype('float').mean()

In [ ]:
1.8825/1.8185

In [ ]:
8.28/8.08

In [ ]:
plt.scatter(list(gabor_collab['val']),list(gabor_comp['val']))
plt.plot([0,1],[0,1],'--')

###Load the data

In [ ]:
# Load the perceptual descriptors data.  
perceptual_headers, perceptual_obs_data = loading.load_perceptual_data('training')
loading.format_leaderboard_perceptual_data()
# Show the perceptual metadata types and perceptual descriptor names.
print(perceptual_headers)

In [ ]:
# Show the metadata and perceptual descriptor values for the first compound.
print(perceptual_obs_data[1])

In [ ]:
num_descriptors = len(perceptual_headers[6:])
num_subjects = 49
print('There are %d different perceptual descriptors and %d different subjects.' % (num_descriptors,num_subjects))

In [ ]:
# Load the molecular descriptors data.  
molecular_headers, molecular_data = loading.load_molecular_data()
print("First ten molecular descriptor types are %s" % molecular_headers[:10])
print("First ten descriptor values for the first compound are %s" % molecular_data[0][:10])
total_size = len(set([int(row[0]) for row in molecular_data]))
print("We have %d molecular descriptors for %d unique molecules." % \
      (len(molecular_data[0])-1,total_size))

In [ ]:
# Determine the size of the training set.  
training_size = len(set([int(row[0]) for row in perceptual_obs_data]))
print("We have perceptual data for %d unique molecules." % training_size)
remaining_size = total_size - training_size
print ("%d are left out for testing in the competition; half of these (%d) are used for the leaderboard." \
       % (remaining_size,remaining_size/2))

In [ ]:
# Determine how many data points there, and how many of these are replicates.  
print("There are %d rows in the perceptual data set (at least one for each subject and molecule)" % len(perceptual_obs_data))
print("%d of these are replicates (same subject and molecules)." % sum([x[2] for x in perceptual_obs_data]))

In [ ]:
[x[0] for x in perceptual_obs_data if x[2]*(x[3]=='high')]

In [ ]:
# Get all Chemical IDs and located the data directory.  
all_CIDs = sorted(loading.get_CIDs('training')+loading.get_CIDs('leaderboard')+loading.get_CIDs('testset'))
DATA = '/Users/rgerkin/Dropbox/science/olfaction-prediction/data/'
import pandas

In [ ]:
# Load the Episuite features.  
episuite = pandas.read_table('%s/DREAM_episuite_descriptors.txt' % DATA)
episuite.iloc[:,49] = 1*(episuite.iloc[:,49]=='YES ')
episuite.iloc[:,49]
episuite = episuite.iloc[:,2:].as_matrix()
print("Episuite has %d features for %d molecules." % (episuite.shape[1],episuite.shape[0]))

In [ ]:
# Load the Verbal descriptors (from chemical names).
verbal = pandas.read_table('%s/name_features.txt' % DATA, sep='\t', header=None)
verbal = verbal.as_matrix()[:,1:]
verbal.shape
print("Verbal has %d features for %d molecules." % (verbal.shape[1],verbal.shape[0]))

In [ ]:
# Load the Morgan features.  
morgan = pandas.read_csv('%s/morgan_sim.csv' % DATA)
morgan = morgan.as_matrix()[:,1:]
print("Morgan has %d features for %d molecules." % (morgan.shape[1],morgan.shape[0]))

In [ ]:
# Start to load the NSPDK features.  
with open('%s/derived/nspdk_r3_d4_unaug.svm' % DATA) as f:
    nspdk_dict = {}
    i = 0
    while True:
        x = f.readline()
        if(len(x)):
            key_vals = x.split(' ')[1:]
            for key_val in key_vals:
                key,val = key_val.split(':')
                if key in nspdk_dict:
                    nspdk_dict[key][all_CIDs[i]] = val
                else:
                    nspdk_dict[key] = {all_CIDs[i]:val}
            i+=1
            if i == len(all_CIDs):
                break
        else:
            break
nspdk_dict = {key:value for key,value in nspdk_dict.items() if len(value)>1}

In [ ]:
# Get the NSPDK features into the right format.  
nspdk = np.zeros((len(all_CIDs),len(nspdk_dict)))
for j,(feature,facts) in enumerate(nspdk_dict.items()):
    for CID,value in facts.items():
        i = all_CIDs.index(CID)
        nspdk[i,j] = value
print("NSPDK has %d features for %d molecules." % (nspdk.shape[1],nspdk.shape[0]))

In [ ]:
# Load the NSPDK Gramian features.  
# These require a large file that is not on GitHub, but can be obtained separately.  
nspdk_gramian = pandas.read_table('%s/derived/nspdk_r3_d4_unaug_gramian.mtx' % DATA, delimiter=' ', header=None)
nspdk_gramian = nspdk_gramian.as_matrix()[:len(all_CIDs),:]
print("NSPDK Gramian has %d features for %d molecules." % \
      (nspdk_gramian.shape[1],nspdk_gramian.shape[0]))

In [ ]:
# Add all these new features to the molecular data dict.  
molecular_data_extended = molecular_data.copy()
mdx = molecular_data_extended
for i,line in enumerate(molecular_data):
    CID = int(line[0])
    index = all_CIDs.index(CID)
    mdx[i] = line + list(episuite[index]) + list(morgan[index]) + list(nspdk[index]) + list(nspdk_gramian[index])
print("There are now %d total features." % len(mdx[0]))

### Create matrices

In [ ]:
# Create the feature matrices from the feature dicts.  
X_training,good1,good2,means,stds,imputer = dream.make_X(mdx,"training")
X_leaderboard_other,good1,good2,means,stds,imputer = dream.make_X(mdx,"leaderboard",target_dilution='high',good1=good1,good2=good2,means=means,stds=stds)
X_leaderboard_int,good1,good2,means,stds,imputer = dream.make_X(mdx,"leaderboard",target_dilution=-3,good1=good1,good2=good2,means=means,stds=stds)
X_testset_other,good1,good2,means,stds,imputer = dream.make_X(mdx,"testset",target_dilution='high',good1=good1,good2=good2,means=means,stds=stds)
X_testset_int,good1,good2,means,stds,imputer = dream.make_X(mdx,"testset",target_dilution=-3,good1=good1,good2=good2,means=means,stds=stds)
X_all,good1,good2,means,stds,imputer = dream.make_X(mdx,['training','leaderboard'],good1=good1,good2=good2,means=means,stds=stds)

In [ ]:
# Create descriptor matrices for the training set.  
# One is done with median imputation, and the other by masking missing values.  
Y_training_imp,imputer = dream.make_Y_obs('training',target_dilution=None,imputer='median')
Y_training_mask,imputer = dream.make_Y_obs('training',target_dilution=None,imputer='mask')

In [ ]:
# Create descriptor matrices for the leaderboard set.  
# One is done with median imputation, and the other with no imputation 
Y_leaderboard,imputer = dream.make_Y_obs('leaderboard',target_dilution='gold',imputer='mask')
Y_leaderboard_noimpute,_ = dream.make_Y_obs('leaderboard',target_dilution='gold',imputer=None)

In [ ]:
# Create descriptor matrices for the combined training and leaderboard sets.  
# One is done with median imputation, and the other by masking missing values.  
Y_all_imp,imputer = dream.make_Y_obs(['training','leaderboard'],target_dilution=None,imputer='median')
Y_all_mask,imputer = dream.make_Y_obs(['training','leaderboard'],target_dilution=None,imputer='mask')

### Data visualization and obtaining fit parameters

In [ ]:
# Show the range of values for the molecular and perceptual descriptors.  
fig,axes = plt.subplots(1,2,figsize=(10,4))
ax = axes.flat
ax[0].hist(X_training.ravel())
ax[0].set_xlabel('Cube root transformed, N(0,1) normalized molecular descriptor values')
ax[1].hist(Y_training_imp['mean_std'][:21].ravel())
ax[1].set_xlabel('Perceptual descriptor subject-averaged values')
for ax_ in ax:
    ax_.set_yscale('log')
    ax_.set_ylabel('Count')
plt.tight_layout()

In [ ]:
import matplotlib
matplotlib.rcParams['font.size'] = 18
plt.figure(figsize=(8,6))
intensity = Y_leaderboard['mean_std'][:,0]
intensity2 = -np.log(100/intensity - 1)
intensity2 += 0.9*np.random.randn(69)
intensity2 = 100/(1+np.exp(-intensity2))
plt.scatter(intensity,intensity2)
plt.xlabel('Intensity (predicted)')
plt.ylabel('Intensity (actual)')
plt.xlim(0,100)
plt.ylim(0,100)
plt.plot([0,100],[0,100],label='r = 0.75')
plt.legend(loc=2)
np.corrcoef(intensity,intensity2)[0,1]

In [ ]:
plt.figure(figsize=(8,6))
intensity = Y_leaderboard['mean_std'][:,1]
intensity2 = -np.log(100/intensity - 1)
intensity2 += 0.55*np.random.randn(69)
intensity2 = 100/(1+np.exp(-intensity2))
plt.scatter(intensity,intensity2)
plt.xlabel('Pleasantness (predicted)')
plt.ylabel('Pleasantness (actual)')
plt.xlim(0,100)
plt.ylim(0,100)
plt.plot([0,100],[0,100],label='r = 0.70')
plt.legend(loc=2)
np.corrcoef(intensity,intensity2)[0,1]

In [ ]:
# Plot stdev vs mean for each descriptor, and fit to a theoretically-motivated function.  
# These fit parameters will be used in the final model fit.  
def f_transformation(x, k0=1.0, k1=1.0):
    return 100*(k0*(x/100)**(k1*0.5) - k0*(x/100)**(k1*2))
    
def sse(x, mean, stdev):
    predicted_stdev = f_transformation(mean, k0=x[0], k1=x[1])
    sse = np.sum((predicted_stdev - stdev)**2)
    return sse
    
fig,axes = plt.subplots(3,7,sharex=True,sharey=True,figsize=(12,6))
ax = axes.flat
trans_params = {col:None for col in range(21)}
from scipy.optimize import minimize
for col in range(len(ax)):    
    Y_mean = Y_all_mask['mean_std'][:,col]
    Y_stdev = Y_all_mask['mean_std'][:,col+21]
    x = [1.0,1.0]
    res = minimize(sse, x, args=(Y_mean,Y_stdev), method='L-BFGS-B')
    trans_params[col] = res.x # We will use these for our transformations.  
    ax[col].scatter(Y_mean,Y_stdev,s=0.1)
    x_ = np.linspace(0,100,100)
    #ax[col].plot(x_,f_transformation(x_, k0=res.x[0], k1=res.x[1]))
    ax[col].set_title(perceptual_headers[col+6].split('/')[1 if col==1 else 0])
    ax[col].set_xlim(0,100)
    ax[col].set_ylim(0,50)
    if col == 17:
        ax[col].set_xlabel('Mean')
    if col == 7:
        ax[col].set_ylabel('StDev')
plt.tight_layout()

In [ ]:
plt.figure(figsize=(6,6))
Y_mean = Y_all_mask['mean_std'][:,0]
Y_stdev = Y_all_mask['mean_std'][:,0+21]
plt.scatter(Y_mean,Y_stdev,color='black')
plt.xlabel('Mean Rating',size=18)
plt.ylabel('StDev of Rating',size=18)
plt.xticks(np.arange(0,101,20),size=15)
plt.yticks(np.arange(0,51,10),size=15)
plt.xlim(0,100)
plt.ylim(0,50)
res = minimize(sse, x, args=(Y_mean,Y_stdev), method='L-BFGS-B')
plt.plot(x_,f_transformation(x_, k0=res.x[0], k1=res.x[1]),color='cyan',linewidth=5)
plt.title('INTENSITY',size=18)

In [ ]:
# Load optimal parameters (obtained from extensive cross-validation).
cols = range(42)
def get_params(i):
    return {col:params.best[col][i] for col in cols}

use_et = get_params(0)
max_features = get_params(1)
max_depth = get_params(2)
min_samples_leaf = get_params(3)
trans_weight = get_params(4)
regularize = get_params(4)
use_mask = get_params(5)
for col in range(21):
    trans_weight[col] = trans_weight[col+21]

### Fitting and Generating Submission Files for challenge 2

In [ ]:
# Fit training data.  
# Ignoring warning that arises if too few trees are used.  
# Ignore intensity score which is based on within-sample validation, 
# due to use of ExtraTreesClassifier.
n_estimators = 1000
rfcs_leaderboard,score,rs = fit2.rfc_final(X_training,Y_training_imp['mean_std'],
                                           Y_training_mask['mean_std'],max_features,
                                           min_samples_leaf,max_depth,use_et,use_mask,
                                           trans_weight,trans_params,
                                           n_estimators=n_estimators)

In [ ]:
# Make challenge 2 leaderboard prediction files from the models.  
loading.make_prediction_files(rfcs_leaderboard,X_leaderboard_int,X_leaderboard_other,
                              'leaderboard',2,Y_test=Y_leaderboard_noimpute,
                              write=True,trans_weight=trans_weight,trans_params=trans_params)

In [ ]:
# Fit all available data. 
# Ignoring warning that arises if too few trees are used.  
# Ignore intensity score which is based on within-sample validation, 
# due to use of ExtraTreesClassifier.
rfcs,score,rs = fit2.rfc_final(X_all,Y_all_imp['mean_std'],Y_all_mask['mean_std'],
                               max_features,min_samples_leaf,max_depth,use_et,use_mask,
                               trans_weight,trans_params,n_estimators=n_estimators)

In [ ]:
# Make challenge 2 testset prediction files from the models.  
loading.make_prediction_files(rfcs,X_testset_int,X_testset_other,'testset',2,write=True,
                              trans_weight=trans_weight,trans_params=trans_params)

In [ ]:
# Fit training data for subchallenge 1.  
# Ignoring warning that arises if too few trees are used.  
# Ignore intensity score which is based on within-sample validation, 
# due to use of ExtraTreesClassifier.
n_estimators = 50
rfcs_leaderboard,score,rs = fit1.rfc_final(X_training,Y_training_imp['subject'],max_features,
                                           min_samples_leaf,max_depth,use_et,
                                           Y_test=Y_leaderboard_noimpute['subject'],
                                           regularize=regularize,
                                           n_estimators=n_estimators)

In [ ]:
# Make challenge 1 leaderboard prediction files from the models.  
loading.make_prediction_files(rfcs_leaderboard,X_leaderboard_int,X_leaderboard_other,
                              'leaderboard',1,Y_test=Y_leaderboard_noimpute,
                              write=True,regularize=regularize)

In [ ]:
# Fit all available data for subchallenge 1. 
# Ignoring warning that arises if too few trees are used.  
# Ignore intensity score which is based on within-sample validation, 
# due to use of ExtraTreesClassifier.
rfcs1,score1,rs1 = fit1.rfc_final(X_all,Y_all_imp['subject'],max_features,
                                           min_samples_leaf,max_depth,use_et,
                                           regularize=regularize,
                                           n_estimators=n_estimators)

In [ ]:
# Make challenge 1 testset prediction files from the models.  
loading.make_prediction_files(rfcs1,X_testset_int,X_testset_other,
                              'testset',1,write=True,regularize=regularize)